# LeetCode 2266 — Count Number of Texts bằng Python

Link đề: https://leetcode.com/problems/count-number-of-texts/ · Độ khó: **Medium** · Chủ đề: Hash Table, Math, String, Dynamic Programming

Skeleton LeetCode:

```python
class Solution:
    def countTexts(self, pressedKeys: str) -> int:
        ...
```

## 1. Mục tiêu học

1. Mô hình hóa bài "giải mã chuỗi phím bấm" thành **đếm số cách chia** một chuỗi thành các khối hợp lệ — họ hàng của LC 91 *Decode Ways*.
2. Nhận ra các **đoạn chữ số giống nhau liên tiếp** (run) là **độc lập** ⇒ đáp án = **tích** số cách của từng đoạn.
3. Số cách chia một đoạn dài `L` thành các khối dài 1..3 (hoặc 1..4 với phím 7, 9) là dãy **Tribonacci / Tetranacci**.
4. Cài đặt: vét cạn đệ quy, DP trên cả chuỗi, gom đoạn + bảng tính sẵn, DP hai dãy song song; làm việc **modulo 10⁹ + 7**.
5. Biết thêm **lũy thừa ma trận** khi độ dài đoạn cực lớn.

## 2. Đề bài & ví dụ

Bàn phím điện thoại: `2: abc`, `3: def`, `4: ghi`, `5: jkl`, `6: mno`, `7: pqrs`, `8: tuv`, `9: wxyz`. Muốn gõ chữ thứ `i` trên một phím thì bấm phím đó `i` lần (ví dụ `'s'` = bấm `7` bốn lần, `'k'` = bấm `5` hai lần).

Bob nhận được chuỗi phím đã bấm `pressedKeys` (không có dấu ngắt giữa các chữ). Ví dụ `"bob"` → `"2266622"`.

Trả về **số tin nhắn** Alice có thể đã gửi, modulo `10⁹ + 7`.

**Ví dụ 1:** `"22233"` → `8`: `"aaadd", "abdd", "badd", "cdd", "aaae", "abe", "bae", "ce"`.

**Ví dụ 2:** `"222222222222222222222222222222222222"` (36 chữ số 2) → `82876089` (= 2082876103 mod 10⁹+7).

**Ràng buộc:** `1 ≤ len(pressedKeys) ≤ 10⁵`, chỉ gồm chữ số `'2'..'9'`.

In [1]:
from itertools import groupby
from functools import lru_cache
import random, time

MOD = 10**9 + 7
KEYS = {"2": "abc", "3": "def", "4": "ghi", "5": "jkl", "6": "mno", "7": "pqrs", "8": "tuv", "9": "wxyz"}
examples = [("22233", 8), ("2" * 36, 82876089)]

def encode(msg):
    out = []
    for ch in msg:
        for d, letters in KEYS.items():
            if ch in letters: out.append(d * (letters.index(ch) + 1))
    return "".join(out)
print(encode("bob"), encode("ce"), encode("aaadd"))

2266622 22233 22233


## 3. Kiến thức nền

### 3.1 `itertools.groupby` — tách các đoạn ký tự giống nhau liên tiếp

In [2]:
s = "2226666377"
print([(k, len(list(g))) for k, g in groupby(s)])

[('2', 3), ('6', 4), ('3', 1), ('7', 2)]


### 3.2 Quy tắc nhân cho các phần độc lập

Nếu chuỗi gồm các phần mà lựa chọn ở phần này **không ảnh hưởng** phần kia, tổng số cách = tích số cách mỗi phần. Ở đây một chữ cái luôn là các lần bấm **cùng một phím liên tiếp** ⇒ không chữ nào "vắt" qua hai đoạn khác phím.

### 3.3 Dãy Tribonacci và Tetranacci

`T(L) = T(L−1) + T(L−2) + T(L−3)`, `T(0) = 1` (một cách: chia rỗng). Đếm số cách viết `L` thành tổng có thứ tự các số 1, 2, 3. Tương tự Tetranacci với 1..4.

In [3]:
def tri(n):
    f = [1] + [0] * n
    for i in range(1, n + 1):
        f[i] = sum(f[i - j] for j in (1, 2, 3) if i - j >= 0)
    return f
def tetra(n):
    f = [1] + [0] * n
    for i in range(1, n + 1):
        f[i] = sum(f[i - j] for j in (1, 2, 3, 4) if i - j >= 0)
    return f
print("tri  :", tri(10)); print("tetra:", tetra(10))

tri  : [1, 1, 2, 4, 7, 13, 24, 44, 81, 149, 274]
tetra: [1, 1, 2, 4, 8, 15, 29, 56, 108, 208, 401]


### 3.4 Số học modulo

`(a · b) mod M = ((a mod M) · (b mod M)) mod M`, tương tự với phép cộng ⇒ có thể lấy mod **sau mỗi bước** để số luôn nhỏ. Python không tràn số, nhưng số rất lớn làm phép tính chậm (và các ngôn ngữ khác tràn).

In [4]:
a, b = 10**18 + 3, 10**18 + 7
print((a * b) % MOD == ((a % MOD) * (b % MOD)) % MOD)

True


## 4. Ý tưởng nền tảng / trực giác

Xét một đoạn `k` chữ số giống nhau, ví dụ `"222"`. Mỗi chữ cái dùng 1, 2 hoặc 3 lần bấm (phím `2` có 3 chữ). Vậy số tin nhắn từ đoạn này = số cách chia 3 thành dãy các phần từ `{1, 2, 3}`: `1+1+1, 1+2, 2+1, 3` ⇒ **4** (= `aaa, ab, ba, c`).

Đoạn `"33"`: `1+1, 2` ⇒ **2** (`dd, e`).

Ví dụ 1 `"22233"` = `"222"` + `"33"` ⇒ `4 × 2 = 8`. ✔

Với phím `7` hoặc `9` (4 chữ), mỗi chữ dùng tới 4 lần bấm ⇒ dãy Tetranacci.

Bảng số cách cho đoạn dài `L`:

| L | 1 | 2 | 3 | 4 | 5 | 6 |
|---|---|---|---|---|---|---|
| phím 3 chữ (Tribonacci) | 1 | 2 | 4 | 7 | 13 | 24 |
| phím 4 chữ (Tetranacci) | 1 | 2 | 4 | 8 | 15 | 29 |

In [5]:
def all_messages(run_digit, L):
    # liệt kê mọi cách chia một đoạn L lần bấm phím run_digit
    letters = KEYS[run_digit]; out = []
    def go(rem, cur):
        if rem == 0: out.append(cur); return
        for j in range(1, min(len(letters), rem) + 1):
            go(rem - j, cur + letters[j - 1])
    go(L, ""); return out

print(all_messages("2", 3), all_messages("3", 2))
print("7777 ->", all_messages("7", 4), len(all_messages("7", 4)))

['aaa', 'ab', 'ba', 'c'] ['dd', 'e']
7777 -> ['pppp', 'ppq', 'pqp', 'pr', 'qpp', 'qq', 'rp', 's'] 8


### 4.1 Số tin nhắn tăng nhanh cỡ nào?

Tỉ số `f(L+1)/f(L)` hội tụ về nghiệm lớn nhất của phương trình đặc trưng: `x³ = x² + x + 1` (≈ 1,8393, "hằng số Tribonacci") và `x⁴ = x³ + x² + x + 1` (≈ 1,9276). Vì vậy số tin nhắn **tăng theo hàm mũ** theo độ dài — với 36 chữ số `2` đã vượt 2·10⁹, lý do đề yêu cầu modulo.

In [6]:
f3 = tri(60); f4 = tetra(60)
print("tỉ số Tribonacci :", [round(f3[i + 1] / f3[i], 4) for i in (5, 10, 20, 50)])
print("tỉ số Tetranacci :", [round(f4[i + 1] / f4[i], 4) for i in (5, 10, 20, 50)])
print("f3(36) =", f3[36], "> 2^31 =", 2**31, "| f3(36) mod p =", f3[36] % MOD)

tỉ số Tribonacci : [1.8462, 1.8394, 1.8393, 1.8393]
tỉ số Tetranacci : [1.9333, 1.9277, 1.9276, 1.9276]
f3(36) = 2082876103 > 2^31 = 2147483648 | f3(36) mod p = 82876089


## 5. Cách 1 — Brute Force / baseline: liệt kê mọi cách giải mã

**Ý tưởng:** đệ quy từ vị trí `i`: chọn độ dài chữ tiếp theo `j = 1..(3 hoặc 4)` sao cho `j` ký tự tiếp theo đều là cùng một chữ số; đếm số đường đi tới cuối chuỗi. Không ghi nhớ ⇒ hàm mũ.

In [7]:
def brute(s):
    n = len(s)
    def go(i):
        if i == n: return 1
        d = s[i]; lim = 4 if d in "79" else 3; total = 0; j = i
        while j < n and j - i < lim and s[j] == d:
            j += 1
            total += go(j)
        return total
    return go(0) % MOD

for s, exp in examples[:1]:
    got = brute(s)
    print(got, "OK" if got == exp else "SAI")

8 OK


**Độ phức tạp:** số lời gọi bằng số cách giải mã ~ `1,84ᴸ` (Tribonacci) — với 36 chữ số `2` đã là ~2·10⁹ lời gọi ⇒ không chạy nổi ví dụ 2.

**Nút thắt:** `go(i)` được gọi lại nhiều lần với cùng `i` (bài toán con chồng lấp) ⇒ ghi nhớ / DP.

In [8]:
def count_calls(s):
    n = len(s); calls = [0]
    def go(i):
        calls[0] += 1
        if i == n: return 1
        d = s[i]; lim = 4 if d in "79" else 3; t = 0; j = i
        while j < n and j - i < lim and s[j] == d:
            j += 1; t += go(j)
        return t
    go(0); return calls[0]

for L in [10, 15, 20]:
    print(f"'2' * {L}: {count_calls('2' * L)} lời gọi")

'2' * 10: 600 lời gọi
'2' * 15: 12640 lời gọi
'2' * 20: 266079 lời gọi


## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (DP tiền tố):** `dp[i]` = số tin nhắn cho tiền tố dài `i`. Chữ cái cuối cùng dùng `j` lần bấm cuối (`j ≤ 3` hoặc `≤ 4` tùy phím) **nếu** `j` ký tự cuối giống nhau:

`dp[i] = Σ_{j} dp[i − j]` với `1 ≤ j ≤ lim(s[i−1])` và `s[i−j..i−1]` cùng chữ số. `dp[0] = 1`.

**Insight 2 (tách đoạn):** chữ cái không bao giờ gồm hai chữ số khác nhau ⇒ ranh giới giữa hai đoạn khác phím **luôn** là ranh giới chữ ⇒ các đoạn độc lập ⇒ đáp án = `Π f_{phím}(độ dài đoạn)` với `f` là Tribonacci (3 chữ) hoặc Tetranacci (4 chữ).

**Insight 3 (tính sẵn):** độ dài đoạn ≤ 10⁵ ⇒ tính sẵn hai dãy tới 10⁵ (mod) một lần, mỗi đoạn tra O(1).

Quy tắc:
- **Nếu** phím là `7` hoặc `9` **thì** dùng dãy 4 số hạng; **ngược lại** dãy 3 số hạng.
- Lấy mod sau mỗi phép cộng/nhân.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Đệ quy có nhớ trên vị trí

In [9]:
def memo(s):
    n = len(s)
    @lru_cache(None)
    def go(i):
        if i == n: return 1
        d = s[i]; lim = 4 if d in "79" else 3; t = 0; j = i
        while j < n and j - i < lim and s[j] == d:
            j += 1; t += go(j)
        return t % MOD
    if n > 500:                      # tránh đệ quy sâu khi demo
        return None
    return go(0)

for s, exp in examples:
    got = memo(s)
    print(got, "OK" if got == exp else "SAI")

8 OK
82876089 OK


Độ sâu đệ quy tới `n` ⇒ chỉ dùng cho chuỗi ngắn (ta chặn `n > 500` trong demo). O(n) thời gian sau khi ghi nhớ.

### 7.2 Cách 3 — DP tiền tố trên cả chuỗi (lặp)

In [10]:
def dp_prefix(s):
    n = len(s); dp = [0] * (n + 1); dp[0] = 1
    for i in range(1, n + 1):
        d = s[i - 1]; lim = 4 if d in "79" else 3
        total = 0
        for j in range(1, lim + 1):
            if i - j < 0 or s[i - j] != d: break      # j ký tự cuối phải cùng chữ số
            total += dp[i - j]
        dp[i] = total % MOD
    return dp[n]

for s, exp in examples:
    got = dp_prefix(s)
    print(got, "OK" if got == exp else "SAI")

8 OK
82876089 OK


`break` ngay khi gặp ký tự khác: nếu `s[i−j] ≠ d` thì mọi `j` lớn hơn cũng không hợp lệ (khối phải liên tục cùng chữ số).

**Độ phức tạp cách 3:** O(4n) thời gian, O(n) bộ nhớ (có thể O(1) với 4 biến cuộn).

### 7.3 Cách 4 — Gom đoạn + hai dãy tính sẵn (cách doocs)

In [11]:
N = 10**5
F3 = [1, 1, 2, 4]; F4 = [1, 1, 2, 4]
for i in range(4, N + 1):
    F3.append((F3[-1] + F3[-2] + F3[-3]) % MOD)
    F4.append((F4[-1] + F4[-2] + F4[-3] + F4[-4]) % MOD)

def runs_precomputed(s):
    ans = 1
    for d, g in groupby(s):
        L = sum(1 for _ in g)
        ans = ans * (F4[L] if d in "79" else F3[L]) % MOD
    return ans

for s, exp in examples:
    got = runs_precomputed(s)
    print(got, "OK" if got == exp else "SAI")
print("F3[:8] =", F3[:8], "F4[:8] =", F4[:8])

8 OK
82876089 OK
F3[:8] = [1, 1, 2, 4, 7, 13, 24, 44] F4[:8] = [1, 1, 2, 4, 8, 15, 29, 56]


**Độ phức tạp cách 4:** tiền xử lý O(N) một lần (dùng chung cho mọi test), mỗi chuỗi O(n). Bộ nhớ O(N).

### 7.4 Cách 5 — DP cuộn O(1) bộ nhớ, reset ở đầu mỗi đoạn

Không cần mảng: giữ 4 giá trị gần nhất của dãy **trong đoạn hiện tại**; khi sang đoạn mới, nhân kết quả đoạn cũ vào `ans` và reset.

In [12]:
def rolling_runs(s):
    ans = 1; i = 0; n = len(s)
    while i < n:
        j = i
        while j < n and s[j] == s[i]: j += 1
        L = j - i; lim = 4 if s[i] in "79" else 3
        w = [1, 0, 0, 0]                   # w[0] = f(t), w[1] = f(t-1), ...
        for _ in range(L):
            nxt = sum(w[:lim]) % MOD
            w = [nxt] + w[:3]
        ans = ans * w[0] % MOD
        i = j
    return ans

for s, exp in examples:
    print(rolling_runs(s), "OK" if rolling_runs(s) == exp else "SAI")

8 OK
82876089 OK


**Độ phức tạp cách 5:** O(n) thời gian, O(1) bộ nhớ phụ.

### 7.5 Cách 6 — Lũy thừa ma trận cho đoạn rất dài

`f(L)` là số hạng của truy hồi tuyến tính bậc 3 (hoặc 4) ⇒ `f(L) = (Aᴸ · v)[0]` với ma trận đồng hành `A`. Lũy thừa nhanh O(k³ log L). Ở bài này không cần (L ≤ 10⁵), nhưng hữu ích cho biến thể input nén (đoạn dài 10¹⁸).

In [13]:
def mat_mul(A, B):
    return [[sum(A[i][t] * B[t][j] for t in range(len(B))) % MOD for j in range(len(B[0]))] for i in range(len(A))]

def mat_pow(A, e):
    R = [[int(i == j) for j in range(len(A))] for i in range(len(A))]
    while e:
        if e & 1: R = mat_mul(R, A)
        A = mat_mul(A, A); e >>= 1
    return R

def f_matrix(L, k):
    A = [[1] * k] + [[int(j == i) for j in range(k)] for i in range(k - 1)]   # ma trận đồng hành
    R = mat_pow(A, L)
    return R[0][0]                      # với v = (1, 0, ..., 0): f(L) = (A^L)[0][0]

print([f_matrix(L, 3) for L in range(8)], F3[:8])
print([f_matrix(L, 4) for L in range(8)], F4[:8])
print("f3(10^18) mod p =", f_matrix(10**18, 3))

[1, 1, 2, 4, 7, 13, 24, 44] [1, 1, 2, 4, 7, 13, 24, 44]
[1, 1, 2, 4, 8, 15, 29, 56] [1, 1, 2, 4, 8, 15, 29, 56]
f3(10^18) mod p = 538436942


## 8. TRACE từng bước

### 8.1 Trace cách 3 (DP tiền tố) trên `"22233"`

| i | s[i−1] | j hợp lệ | dp[i] |
|---|---|---|---|
| 0 | — | — | 1 |
| 1 | 2 | 1 | dp[0] = 1 |
| 2 | 2 | 1, 2 | dp[1] + dp[0] = 2 |
| 3 | 2 | 1, 2, 3 | 2 + 1 + 1 = 4 |
| 4 | 3 | 1 (s[2] = '2' ≠ '3') | dp[3] = 4 |
| 5 | 3 | 1, 2 | dp[4] + dp[3] = 8 |

In [14]:
def trace_dp(s):
    n = len(s); dp = [0] * (n + 1); dp[0] = 1
    print("dp[0] = 1")
    for i in range(1, n + 1):
        d = s[i - 1]; lim = 4 if d in "79" else 3; terms = []
        for j in range(1, lim + 1):
            if i - j < 0 or s[i - j] != d: break
            terms.append(j)
        dp[i] = sum(dp[i - j] for j in terms) % MOD
        print(f"dp[{i}] (ký tự '{d}'): j ∈ {terms} -> " + " + ".join(f"dp[{i - j}]" for j in terms) + f" = {dp[i]}")
    return dp[n]

trace_dp("22233")

dp[0] = 1
dp[1] (ký tự '2'): j ∈ [1] -> dp[0] = 1
dp[2] (ký tự '2'): j ∈ [1, 2] -> dp[1] + dp[0] = 2
dp[3] (ký tự '2'): j ∈ [1, 2, 3] -> dp[2] + dp[1] + dp[0] = 4
dp[4] (ký tự '3'): j ∈ [1] -> dp[3] = 4
dp[5] (ký tự '3'): j ∈ [1, 2] -> dp[4] + dp[3] = 8


8

### 8.2 Trace cách 4 (gom đoạn) trên `"2266622"` (chuỗi của "bob") và `"777799"`

In [15]:
def trace_runs(s):
    ans = 1
    for d, g in groupby(s):
        L = sum(1 for _ in g); w = F4[L] if d in "79" else F3[L]
        ans = ans * w % MOD
        print(f"đoạn '{d}' x {L}: {'Tetranacci' if d in '79' else 'Tribonacci'}({L}) = {w}, tích = {ans}")
    return ans

print("Kết quả:", trace_runs("2266622")); print()
print("Kết quả:", trace_runs("777799"))

đoạn '2' x 2: Tribonacci(2) = 2, tích = 2
đoạn '6' x 3: Tribonacci(3) = 4, tích = 8
đoạn '2' x 2: Tribonacci(2) = 2, tích = 16
Kết quả: 16

đoạn '7' x 4: Tetranacci(4) = 8, tích = 8
đoạn '9' x 2: Tetranacci(2) = 2, tích = 16
Kết quả: 16


Kiểm chứng `"2266622"`: đoạn `22` (2 cách) × `666` (4 cách) × `22` (2 cách) = 16 — trong đó có `"bob"`.

In [16]:
def decode_all(s):
    n = len(s); out = []
    def go(i, cur):
        if i == n: out.append(cur); return
        d = s[i]; letters = KEYS[d]; j = i
        while j < n and j - i < len(letters) and s[j] == d:
            j += 1; go(j, cur + letters[j - i - 1])
    go(0, ""); return out

msgs = decode_all("2266622")
print(len(msgs), "bob" in msgs, msgs[:6])

16 True ['aammmaa', 'aammmb', 'aamnaa', 'aamnb', 'aanmaa', 'aanmb']


## 9. Code nộp LeetCode

In [17]:
from itertools import groupby

MOD = 10**9 + 7
_N = 10**5
_F3 = [1, 1, 2, 4]          # phím 3 chữ: Tribonacci
_F4 = [1, 1, 2, 4]          # phím 7, 9 (4 chữ): Tetranacci
for _i in range(4, _N + 1):
    _F3.append((_F3[-1] + _F3[-2] + _F3[-3]) % MOD)
    _F4.append((_F4[-1] + _F4[-2] + _F4[-3] + _F4[-4]) % MOD)

class Solution:
    def countTexts(self, pressedKeys: str) -> int:
        ans = 1
        for d, g in groupby(pressedKeys):
            L = sum(1 for _ in g)                       # độ dài đoạn cùng phím
            ans = ans * (_F4[L] if d in "79" else _F3[L]) % MOD
        return ans

In [18]:
sol = Solution()
for s, exp in examples:
    got = sol.countTexts(s)
    print(got, "| kỳ vọng", exp, "| OK" if got == exp else "| SAI")

8 | kỳ vọng 8 | OK
82876089 | kỳ vọng 82876089 | OK


Vì sao dùng `sum(1 for _ in g)` để lấy độ dài? `g` là một **iterator** (không có `len`); đếm bằng generator không tạo list phụ. Có thể viết `len(list(g))` — đơn giản hơn nhưng tốn bộ nhớ tạm bằng độ dài đoạn.

Bảng `_F3`, `_F4` tính **một lần** ở mức module (LeetCode tạo nhiều đối tượng `Solution` cho nhiều test nhưng module chỉ nạp một lần) ⇒ chi phí tiền xử lý không lặp lại.

## 10. Vì sao thuật toán đúng

**Song ánh tin nhắn ↔ cách chia.** Mỗi tin nhắn tương ứng duy nhất một cách chia `pressedKeys` thành các khối liên tiếp, mỗi khối gồm `t` lần cùng một chữ số với `1 ≤ t ≤ (số chữ trên phím)`; ngược lại mỗi cách chia như vậy cho đúng một tin nhắn (khối `t` lần phím `d` = chữ thứ `t` của `d`).

**Độc lập giữa các đoạn.** Một khối không thể chứa hai chữ số khác nhau ⇒ mọi ranh giới giữa hai đoạn khác phím là ranh giới khối ⇒ một cách chia toàn chuỗi = bộ cách chia của từng đoạn ⇒ số cách = tích.

**Số cách chia một đoạn dài L** với khối dài `1..k`: chọn độ dài khối cuối `j` ⇒ `f(L) = Σ_{j=1..min(k,L)} f(L − j)`, `f(0) = 1` — đúng truy hồi Tribonacci (k = 3) / Tetranacci (k = 4) với `f(1) = 1, f(2) = 2, f(3) = 4`. ∎

## 11. Phân tích Time / Space Complexity

`n ≤ 10⁵`.

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn | O(c^n), c ≈ 1,84–1,93 | O(n) |
| 2. Đệ quy có nhớ | O(n) | O(n) + đệ quy sâu |
| 3. DP tiền tố | O(4n) | O(n) |
| 4. Gom đoạn + bảng | O(N) một lần + O(n) | O(N) |
| 5. DP cuộn | O(n) | O(1) |
| 6. Lũy thừa ma trận | O(số đoạn · k³ log L) | O(k²) |

In [19]:
rng = random.Random(2266)
big = "".join(rng.choice("23456789") * rng.randint(1, 30) for _ in range(6000))[:10**5]
for name, f in [("dp_prefix", dp_prefix), ("runs_precomputed", runs_precomputed), ("rolling_runs", rolling_runs),
                ("Solution", Solution().countTexts)]:
    t = time.perf_counter(); r = f(big); dt = time.perf_counter() - t
    print(f"{name:<17} -> {r}  {dt*1000:.0f} ms")

dp_prefix         -> 652696650  32 ms
runs_precomputed  -> 652696650  3 ms
rolling_runs      -> 652696650  18 ms
Solution          -> 652696650  3 ms


## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [20]:
ALL = {"brute": brute, "memo": memo, "dp_prefix": dp_prefix, "runs_precomputed": runs_precomputed,
       "rolling_runs": rolling_runs, "Solution": Solution().countTexts}
edge = examples + [
    ("2", 1), ("7", 1),
    ("22", 2), ("222", 4), ("2222", 7),
    ("7777", 8), ("77777", 15),
    ("23", 1),                              # hai phím khác nhau
    ("2266622", 16),
    ("9" * 10, F4[10]),
    ("2" * (10**5), F3[10**5]),
]
for s, exp in edge:
    for name, f in ALL.items():
        if name == "brute" and len(s) > 20: continue
        got = f(s)
        if got is None: continue                    # memo bỏ qua chuỗi dài
        assert got == exp, (name, s[:20], got, exp)
print("Edge case OK:", len(edge))

Edge case OK: 13


### 12.2 Đối chiếu ngẫu nhiên với vét cạn và với liệt kê tin nhắn thật

In [21]:
rng = random.Random(42)
for _ in range(1500):
    s = "".join(rng.choice(rng.choice(["2", "27", "79", "23456789"])) for _ in range(rng.randint(1, 14)))
    ref = brute(s)
    for name, f in ALL.items():
        got = f(s)
        if got is not None: assert got == ref, (name, s)
    if len(s) <= 10:
        msgs = decode_all(s)
        assert len(msgs) == ref and len(set(msgs)) == len(msgs)
        assert all(encode(m) == s for m in msgs)
print("1500 case ngẫu nhiên OK (kể cả kiểm tra mã hóa ngược)")

1500 case ngẫu nhiên OK (kể cả kiểm tra mã hóa ngược)


### 12.3 Tính chất: đáp án không phụ thuộc thứ tự các đoạn; nối hai chuỗi có ký tự đầu/cuối khác nhau ⇒ tích

In [22]:
rng = random.Random(7)
for _ in range(300):
    a = "".join(rng.choice("2357") for _ in range(rng.randint(1, 20)))
    b = "".join(rng.choice("2357") for _ in range(rng.randint(1, 20)))
    if a[-1] != b[0]:
        assert Solution().countTexts(a + b) == Solution().countTexts(a) * Solution().countTexts(b) % MOD
    runs = ["".join(g) for _, g in groupby(a)]
    rng.shuffle(runs)
    joined = "".join(runs)
    if all(x[-1] != y[0] for x, y in zip(runs, runs[1:])):
        assert Solution().countTexts(joined) == Solution().countTexts(a)
print("tính chất OK")

tính chất OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Dùng dãy 3 số hạng cho cả phím 7 và 9

In [23]:
def wrong_all_three(s):
    ans = 1
    for d, g in groupby(s):
        ans = ans * F3[sum(1 for _ in g)] % MOD           # SAI: 7 và 9 có 4 chữ
    return ans

print("bản sai :", wrong_all_three("7777"), "| bản đúng:", Solution().countTexts("7777"))

bản sai : 7 | bản đúng: 8


**Sửa:** `F4` cho `'7'` và `'9'` (`pqrs`, `wxyz`).

### Lỗi 2 — DP tiền tố không kiểm tra các ký tự cuối có **cùng** chữ số

In [24]:
def wrong_no_same_check(s):
    n = len(s); dp = [0] * (n + 1); dp[0] = 1
    for i in range(1, n + 1):
        lim = 4 if s[i - 1] in "79" else 3
        dp[i] = sum(dp[i - j] for j in range(1, lim + 1) if i - j >= 0) % MOD   # SAI
    return dp[n]

print("bản sai :", wrong_no_same_check("23"), "| bản đúng:", Solution().countTexts("23"))

bản sai : 2 | bản đúng: 1


**Vì sao sai:** khối `"23"` không phải một chữ cái. **Sửa:** chỉ cộng `dp[i−j]` khi `s[i−j..i−1]` cùng chữ số (dừng ở ký tự khác đầu tiên).

### Lỗi 3 — Cộng thay vì nhân giữa các đoạn

In [25]:
def wrong_add_runs(s):
    total = 0
    for d, g in groupby(s):
        L = sum(1 for _ in g); total += F4[L] if d in "79" else F3[L]     # SAI
    return total % MOD

print("bản sai :", wrong_add_runs("22233"), "| bản đúng:", Solution().countTexts("22233"))

bản sai : 6 | bản đúng: 8


**Sửa:** các đoạn độc lập ⇒ **nhân** (quy tắc nhân).

### Lỗi 4 — Khởi tạo sai `f(0)` (đặt `f(0) = 0`)

In [26]:
def wrong_base(s):
    def f(L, k):
        a = [0] * (L + 1)                                  # SAI: a[0] = 0
        for i in range(1, L + 1):
            a[i] = sum(a[i - j] for j in range(1, k + 1) if i - j >= 0)
        return a[L]
    ans = 1
    for d, g in groupby(s):
        ans = ans * f(sum(1 for _ in g), 4 if d in "79" else 3) % MOD
    return ans

print("bản sai :", wrong_base("22233"), "| bản đúng:", Solution().countTexts("22233"))

bản sai : 0 | bản đúng: 8


**Vì sao sai:** `f(0) = 1` nghĩa là "có đúng một cách chia chuỗi rỗng" — cơ sở của mọi truy hồi đếm. Đặt 0 làm mọi giá trị bằng 0. **Sửa:** `f(0) = 1`.

### Lỗi 5 — Không lấy mod (số quá lớn, chậm; ngôn ngữ khác tràn)

In [27]:
def no_mod(s):
    ans = 1
    for d, g in groupby(s):
        L = sum(1 for _ in g); k = 4 if d in "79" else 3
        f = [1] + [0] * L
        for i in range(1, L + 1): f[i] = sum(f[i - j] for j in range(1, k + 1) if i - j >= 0)
        ans *= f[L]
    return ans

s = "2" * 3000
t = time.perf_counter(); big_val = no_mod(s); t1 = time.perf_counter() - t
print(f"không mod: số có {big_val.bit_length()} bit, {t1*1000:.0f} ms; quên trả về mod -> {big_val % MOD == Solution().countTexts(s)} sau khi mod")
print("nếu trả thẳng số lớn thì SAI:", big_val == Solution().countTexts(s))

không mod: số có 2637 bit, 2 ms; quên trả về mod -> True sau khi mod
nếu trả thẳng số lớn thì SAI: False


**Sửa:** lấy mod sau mỗi phép cộng/nhân (và trả về giá trị đã mod).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Có bao nhiêu cách giải mã/chia chuỗi" ⇒ DP tiền tố `dp[i] = Σ dp[i − j]` trên các khối hợp lệ kết thúc ở `i` (Decode Ways).
- Khối chỉ gồm ký tự **giống nhau** ⇒ tách theo `groupby`, **nhân** kết quả các đoạn.
- Truy hồi tuyến tính hệ số hằng ⇒ tính sẵn bảng; độ dài khổng lồ ⇒ lũy thừa ma trận.
- Luôn kiểm tra `f(0) = 1` và mod.
- Đếm số **cách** (không phải liệt kê) ⇒ đừng sinh tin nhắn thật; số lượng tăng theo hàm mũ (mục 4.1).
- Bài cùng họ: 91 Decode Ways, 639 Decode Ways II, 70 Climbing Stairs, 1137 N-th Tribonacci Number, 17 Letter Combinations of a Phone Number.

## 15. Biến thể của bài

### 15.1 Biến thể A — Input nén `[(chữ số, số lần), ...]` với số lần tới 10¹⁸

In [28]:
def count_texts_compressed(pairs):
    merged = []
    for d, c in pairs:
        if merged and merged[-1][0] == d: merged[-1][1] += c
        else: merged.append([d, c])
    ans = 1
    for d, c in merged:
        ans = ans * f_matrix(c, 4 if d in "79" else 3) % MOD
    return ans

print(count_texts_compressed([("2", 3), ("3", 2)]), count_texts_compressed([("2", 10**18), ("7", 5)]))
assert count_texts_compressed([("2", 3), ("3", 2)]) == 8
assert count_texts_compressed([("2", 2), ("2", 1), ("3", 2)]) == 8          # gộp đoạn liền kề cùng phím

8 76554074


Lưu ý gộp các cặp liền kề **cùng** chữ số trước (đoạn thật có thể bị chia thành nhiều cặp).

### 15.2 Biến thể B — Có **dấu ngắt** `' '` (Bob nhận được cả khoảng trắng khi Alice dừng)

Khoảng trắng tách đoạn: hai đoạn cùng phím nhưng cách nhau bởi dấu ngắt vẫn độc lập.

In [29]:
def count_with_pauses(s):
    ans = 1
    for part in s.split(" "):
        if part: ans = ans * Solution().countTexts(part) % MOD
    return ans

print(count_with_pauses("222 22"), "(4 x 2)", count_with_pauses("22222"), "(không ngắt: 13)")

8 (4 x 2) 13 (không ngắt: 13)


### 15.3 Biến thể C — Đếm số tin nhắn có **đúng m chữ cái**

Thêm chiều số chữ: `g[L][m]` = số cách chia đoạn dài `L` thành `m` khối (1..k). Nhân các đoạn ⇒ tích chập theo `m`.

In [30]:
def count_exact_letters(s, m):
    total = [1] + [0] * m                         # total[t] = số cách tạo t chữ với các đoạn đã xét
    for d, g in groupby(s):
        L = sum(1 for _ in g); k = 4 if d in "79" else 3
        ways = [[0] * (m + 1) for _ in range(L + 1)]; ways[0][0] = 1
        for i in range(1, L + 1):
            for t in range(1, m + 1):
                ways[i][t] = sum(ways[i - j][t - 1] for j in range(1, k + 1) if i - j >= 0)
        new = [0] * (m + 1)
        for a in range(m + 1):
            if total[a]:
                for b in range(m + 1 - a):
                    new[a + b] = (new[a + b] + total[a] * ways[L][b]) % MOD
        total = new
    return total[m]

msgs = decode_all("22233")
print([count_exact_letters("22233", m) for m in range(1, 6)], [sum(1 for x in msgs if len(x) == m) for m in range(1, 6)])

[0, 1, 3, 3, 1] [0, 1, 3, 3, 1]


### 15.4 Biến thể D — LC 91 *Decode Ways* (`'1'..'26'` → `'A'..'Z'`) cùng khung DP tiền tố

Khối hợp lệ dài 1 (`'1'..'9'`) hoặc dài 2 (`'10'..'26'`); `dp[i] = dp[i−1]·[khối 1 hợp lệ] + dp[i−2]·[khối 2 hợp lệ]`. Cùng khung "khối kết thúc tại `i`" như cách 3, chỉ đổi điều kiện hợp lệ.

In [31]:
def num_decodings(s):
    n = len(s); dp = [0] * (n + 1); dp[0] = 1
    for i in range(1, n + 1):
        if s[i - 1] != "0": dp[i] += dp[i - 1]
        if i >= 2 and "10" <= s[i - 2:i] <= "26": dp[i] += dp[i - 2]
    return dp[n]

def num_decodings_brute(s):
    if not s: return 1
    t = 0
    if s[0] != "0": t += num_decodings_brute(s[1:])
    if len(s) >= 2 and "10" <= s[:2] <= "26": t += num_decodings_brute(s[2:])
    return t

print(num_decodings("12"), num_decodings("226"), num_decodings("06"))
rng = random.Random(154)
for _ in range(500):
    s = "".join(rng.choice("0123456789") for _ in range(rng.randint(1, 12)))
    assert num_decodings(s) == num_decodings_brute(s)
print("Decode Ways: khớp brute force")

2 3 0
Decode Ways: khớp brute force


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | liệt kê | mũ | O(n) | kiểm chứng |
| 2. Đệ quy có nhớ | go(i) | O(n) | O(n) | n nhỏ |
| 3. DP tiền tố | dp[i] = Σ dp[i−j] | O(n) | O(n) | tổng quát (Decode Ways) |
| 4. Gom đoạn + bảng | tích Tribonacci/Tetranacci | O(n) | O(N) | **mặc định** |
| 5. DP cuộn | 4 biến, reset mỗi đoạn | O(n) | O(1) | bộ nhớ chặt |
| 6. Ma trận | lũy thừa nhanh | O(k³ log L) | O(k²) | đoạn dài khổng lồ |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `countTexts("444479999555")` và `countTexts("23456789")`?

**Bài 2 (tính tay):** Liệt kê mọi tin nhắn cho `"7777"`.

**Bài 3 (điền TODO):** Viết `bt3_todo(s)` cài DP tiền tố (cách 3) bằng **4 biến cuộn** (O(1) bộ nhớ) trên cả chuỗi (không tách đoạn).

**Bài 4 (tự viết lại):** Viết `bt4_todo(L, k)` trả `f(L)` (số cách chia đoạn dài `L` thành khối 1..k) mod p bằng lũy thừa ma trận.

**Bài 5 (mini challenge):** Viết `bt5_todo(s)` trả về **tin nhắn nhỏ nhất theo thứ tự từ điển** trong mọi tin nhắn có thể.

In [32]:
def bt3_todo(s):
    # TODO: DP tiền tố với 4 biến cuộn
    pass

def bt4_todo(L, k):
    # TODO: lũy thừa ma trận
    pass

def bt5_todo(s):
    # TODO: tin nhắn nhỏ nhất theo từ điển
    raise NotImplementedError

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: đoạn `4444, 7, 9999, 555`. Mức 2: `F3[4] = 7`, `F4[1] = 1`, `F4[4] = 8`, `F3[3] = 4`. Mức 3: `7·1·8·4 = 224`; chuỗi hai: mọi đoạn dài 1 ⇒ `1`.

**Bài 2.** Mức 1: Tetranacci(4) = 8. Mức 2: các cách chia 4: 1111, 112, 121, 211, 22, 13, 31, 4. Mức 3: `pppp, ppq, pqp, qpp, qq, pr, rp, s`.

**Bài 3.** Mức 1: `dp[i]` chỉ phụ thuộc `dp[i−1..i−4]`. Mức 2: giữ list 4 phần tử `last`. Mức 3: cộng `last[j−1]` khi `s[i−j] == s[i−1]`, rồi dịch cửa sổ.

**Bài 4.** Mức 1: ma trận đồng hành `k × k`. Mức 2: hàng đầu toàn 1, đường chéo phụ 1. Mức 3: `(Aᴸ)[0][0]`.

**Bài 5.** Mức 1: tham lam từ trái: ở mỗi bước nên chọn chữ nhỏ nhất có thể; băn khoăn: chữ đầu phím (1 lần bấm) làm tin nhắn **dài** hơn — có sao không? Mức 2: so sánh từ điển: `"aa" < "b"`, nên luôn chọn `a` (1 lần bấm) là tốt nhất. Mức 3: đáp án = thay mỗi chữ số bằng chữ đầu tiên của phím (`"22233" → "aaadd"`).

## 19. Đáp án tham khảo cho bài tập

**Bài 1:** `224`, `1`. **Bài 2:** `pppp, ppq, pqp, qpp, qq, pr, rp, s`.

In [33]:
assert Solution().countTexts("444479999555") == 224
assert Solution().countTexts("23456789") == 1
assert sorted(decode_all("7777")) == sorted(["pppp", "ppq", "pqp", "qpp", "qq", "pr", "rp", "s"])

def bt3(s):
    last = [1, 0, 0, 0]                    # last[0] = dp[i-1], last[1] = dp[i-2], ...
    for i in range(1, len(s) + 1):
        d = s[i - 1]; lim = 4 if d in "79" else 3; cur = 0
        for j in range(1, lim + 1):
            if i - j < 0 or s[i - j] != d: break
            cur += last[j - 1]
        last = [cur % MOD] + last[:3]
    return last[0]

def bt4(L, k):
    return f_matrix(L, k)

def bt5(s):
    return "".join(KEYS[d][0] for d in s)

rng = random.Random(1966)
for _ in range(1000):
    s = "".join(rng.choice(rng.choice(["2", "79", "2379"])) for _ in range(rng.randint(1, 12)))
    assert bt3(s) == brute(s)
    if len(s) <= 10:
        assert bt5(s) == min(decode_all(s))
for L in range(0, 60):
    assert bt4(L, 3) == F3[L] and bt4(L, 4) == F4[L]
print("Đáp án bài tập: tất cả assert OK")

Đáp án bài tập: tất cả assert OK


Ghi chú bài 5: với hai chuỗi mà một chuỗi là tiền tố của chuỗi kia, chuỗi ngắn hơn nhỏ hơn — nhưng ở đây mọi tin nhắn bắt đầu bằng một chữ của phím đầu tiên; chữ `a`-đầu-phím là nhỏ nhất trên phím đó, và dùng 1 lần bấm thì phần còn lại vẫn bắt đầu bằng chữ cùng phím (cũng nhỏ nhất). Quy nạp cho thấy chọn chữ đầu phím ở mọi bước cho chuỗi nhỏ nhất.

## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
F3 = [1, 1, 2, 4]; F4 = [1, 1, 2, 4]            # f(0) = 1
for i in range(4, N + 1):
    F3.append((F3[-1] + F3[-2] + F3[-3]) % MOD)
    F4.append((F4[-1] + F4[-2] + F4[-3] + F4[-4]) % MOD)
ans = 1
for d, g in groupby(pressedKeys):
    L = sum(1 for _ in g)
    ans = ans * (F4[L] if d in '79' else F3[L]) % MOD
```

**Checklist:**
- [ ] Phím `7`, `9` có **4** chữ.
- [ ] Khối chỉ gồm **cùng** chữ số; các đoạn khác phím **nhân** với nhau.
- [ ] `f(0) = 1` (một cách chia chuỗi rỗng).
- [ ] Lấy mod `10⁹ + 7` mỗi bước.
- [ ] Tính bảng **một lần** (mức module) để không tốn lại cho mỗi test.
- [ ] Đã thử: một ký tự, đoạn dài 10⁵, xen kẽ phím 3 chữ và 4 chữ.